# Baseline per la predizione di `delta_rtt`

*Progetto: tokenizzazione dei network flow CERN (dati Scitags Firefly), Gruppo 1.*

**Scopo.** Fissare una scala di riferimento per valutare la tokenizzazione: dal *pavimento*
(nessuna informazione) al *soffitto pratico* (modelli forti su input non tokenizzati).
Il compito è prevedere `delta_rtt = end_rtt − start_rtt` (ms) al momento dell'inizio del flow.

**Dati.** `flows_prepared.parquet`: flow con `start_rtt > 0` e `end_rtt > 0`, split cronologico
70/15/15 (train fino al 12/10 17:12, validation fino al 13/10 22:18, poi test).

**Regole.**
- Tutto ciò che si stima (mediane, soglie, vocabolari, parametri) si stima **solo sul train**.
- Si confrontano i modelli sulla **validation**; il **test** si usa una sola volta, alla fine del progetto.
- Vietati come input i campi noti solo alla fine del flow: `end_rtt`, durata, colonne `an_*`
  (usate solo per analizzare gli errori).
- Metrica principale: **MAE**; accanto RMSE, RMSE robusto †, P90 e P99 dell'errore assoluto.

**Struttura.** §1 Setup · §2 Valutazione · §3 B0 mediana globale · §4 B1 mediana per tratta ·
§5 Preparazione per i modelli · §6 B4 LightGBM · §7 B2 ridge · §8 B3 rete neurale · §9 Conclusioni.

## 1. Setup

Import, percorsi dei dati, connessione a DuckDB e vista `fp` sul file preparato.
DuckDB interroga il file Parquet direttamente dal disco: in Python arrivano solo i risultati
delle query, non i 67 M di righe.

In [1]:
import json
import sys
import time
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

# --- Percorsi ---
PROJECT_ROOT = Path.cwd().parent                     # il notebook sta in results/
DATA_DIR = PROJECT_ROOT / "data" / "student_dataset"
FLOWS    = DATA_DIR / "flows.parquet"
SPLITS   = DATA_DIR / "splits.json"
PREPARED = DATA_DIR / "flows_prepared.parquet"

for f in [FLOWS, SPLITS, PREPARED]:
    print(f"{f.name:25s} esiste: {f.exists()}")

# --- Connessione DuckDB ---
con = duckdb.connect()
con.execute("SET threads TO 8")
con.execute("SET memory_limit = '8GB'")
con.execute(f"SET temp_directory = '{(PROJECT_ROOT / 'duckdb_tmp').as_posix()}'")
con.execute("SET preserve_insertion_order = false")
con.execute("SET TimeZone = 'UTC'")

con.execute(f"CREATE OR REPLACE VIEW fp AS SELECT * FROM read_parquet('{PREPARED.as_posix()}')")

print("Python", sys.version.split()[0], "| DuckDB", duckdb.__version__,
      "| pandas", pd.__version__, "| numpy", np.__version__)

flows.parquet             esiste: True
splits.json               esiste: True
flows_prepared.parquet    esiste: True
Python 3.14.0 | DuckDB 2.0.0 | pandas 2.3.3 | numpy 2.5.3


### Controllo dei dati

Righe, inizio e fine di ogni split (attesi: train 46.730.686, validation 10.069.254,
test 10.086.352) e numero di colonne del file preparato (attese 41).
Per il test si leggono solo questi conteggi descrittivi, nessun valore del target.

In [2]:
display(con.execute("""
    SELECT split,
           COUNT(*)     AS n,
           MIN(t_start) AS inizio,
           MAX(t_start) AS fine
    FROM fp
    GROUP BY split
    ORDER BY MIN(t_start)
""").df())

colonne = con.execute("DESCRIBE fp").df()
print(f"Colonne: {len(colonne)}")

,split,n,inizio,fine
0,train,46730686,2026-10-07 23:59:59.465736+00:00,2026-10-12 17:12:05.477960+00:00
1,val,10069254,2026-10-12 17:12:05.481692+00:00,2026-10-13 22:18:28.700870+00:00
2,test,10086352,2026-10-13 22:18:28.701549+00:00,2026-10-14 23:59:59.753340+00:00


Colonne: 41


## 2. Valutazione

Tutti i modelli si valutano sulla validation con la stessa funzione e le stesse metriche
(errore $e_i = y_i - \hat y_i$, in ms):

| Metrica | Definizione | Lettura |
|---|---|---|
| **MAE** (principale) | media di $\lvert e_i \rvert$ | errore tipico |
| RMSE | $\sqrt{\text{media di } e_i^2}$ | dominato dagli errori grandi |
| RMSE robusto † | RMSE sui soli flow *tipici* | errori grandi sui flow normali |
| P90, P99 | 90° e 99° percentile di $\lvert e_i \rvert$ | "nel 90 % dei flow sbaglio meno di…" |

### 2.1 Flow tipici (per l'RMSE robusto)

Sulla B0, lo 0,1 % dei flow produce il 98,8 % dell'errore quadratico: l'RMSE completo misura quasi
solo quei pochi casi. Definiamo quindi un insieme **fisso** di flow *tipici*: quelli con `delta_rtt`
tra il quantile 0,05 % e il 99,95 % **del train**. L'insieme dipende dal valore vero, non dalle
previsioni, quindi è lo stesso per tutti i modelli. Ogni tabella riporta sempre entrambi gli RMSE.

In [3]:
Q_LO, Q_HI = 0.0005, 0.9995

LO, HI = con.execute(f"""
    SELECT quantile_cont(delta_rtt, [{Q_LO}, {Q_HI}])
    FROM fp
    WHERE split = 'train'
""").fetchone()[0]

NOTA_ROBUSTO = (f"† RMSE robusto: solo flow con delta_rtt in [{LO:.2f}, {HI:.2f}] ms, "
                f"quantili {Q_LO:.2%}–{Q_HI:.2%} del train (insieme fisso, uguale per tutti i modelli).")

print(f"LO = {LO:.3f} ms | HI = {HI:.3f} ms")

con.execute(f"""
    SELECT split,
           COUNT(*)                                                       AS n,
           COUNT(*) FILTER (WHERE delta_rtt BETWEEN {LO} AND {HI})        AS n_tipici,
           100 * AVG(CASE WHEN delta_rtt BETWEEN {LO} AND {HI} THEN 1 ELSE 0 END) AS pct_tipici
    FROM fp
    WHERE split IN ('train', 'val')
    GROUP BY split
    ORDER BY split
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

LO = -184.233 ms | HI = 58.823 ms


,split,n,n_tipici,pct_tipici
0,train,46730686,46683954,99.899997
1,val,10069254,10061030,99.918326


### 2.2 Funzione di valutazione

`valuta(vista)` riceve il nome di una tabella o vista DuckDB con le righe della validation,
la colonna vera `delta_rtt`, la previsione `pred` e le colonne dei sottogruppi.
Calcola le metriche in DuckDB (in Python arrivano solo i numeri), globalmente e per sottogruppo:

| Sottogruppo | Definizione | Perché |
|---|---|---|
| globale | tutti i flow | metrica ufficiale |
| FNAL | server o client = `USCMS-FNAL-WC1` | target del train alterato da un guasto di telemetria (in indagine) |
| sito mancante | server o client = `MISSING` | ~18 % dei flow senza sito CRIC |
| verso incerto | `verso_incerto` | ruoli server/client assegnati in modo incerto (0,27 %) |
| flow > 1 min | `an_durata_s > 60` | origine del picco a ~5 ms; durata usata **solo per dividere** |

`mostra(ris)` stampa la tabella con nomi leggibili e la nota sull'RMSE robusto.

In [4]:
SOTTOGRUPPI = {
    "globale":       "TRUE",
    "FNAL":          "server_site = 'USCMS-FNAL-WC1' OR client_site = 'USCMS-FNAL-WC1'",
    "sito mancante": "server_site = 'MISSING' OR client_site = 'MISSING'",
    "verso incerto": "verso_incerto",
    "flow > 1 min":  "an_durata_s > 60",
}

def valuta(vista):
    """vista: tabella/vista DuckDB con delta_rtt, pred e le colonne dei sottogruppi (solo validation)."""
    parti = []
    for ordine, (nome, cond) in enumerate(SOTTOGRUPPI.items()):
        parti.append(f"""
            SELECT {ordine} AS ordine, '{nome}' AS gruppo, COUNT(*) AS n,
                   AVG(ABS(delta_rtt - pred))                            AS mae,
                   SQRT(AVG(POWER(delta_rtt - pred, 2)))                 AS rmse,
                   SQRT(AVG(POWER(delta_rtt - pred, 2))
                        FILTER (WHERE delta_rtt BETWEEN {LO} AND {HI})) AS rmse_robusto,
                   quantile_cont(ABS(delta_rtt - pred), 0.90)            AS p90_abs,
                   quantile_cont(ABS(delta_rtt - pred), 0.99)            AS p99_abs
            FROM {vista}
            WHERE {cond}
        """)
    query = "SELECT * FROM (" + " UNION ALL ".join(parti) + ") ORDER BY ordine"
    return con.execute(query).df().drop(columns="ordine")

NOMI_METRICHE = {"mae": "MAE", "rmse": "RMSE", "rmse_robusto": "RMSE robusto †",
                 "p90_abs": "P90 |err|", "p99_abs": "P99 |err|"}

def mostra(ris):
    """Mostra la tabella delle metriche (ms) con nomi leggibili e la nota sull'RMSE robusto."""
    display(ris.rename(columns=NOMI_METRICHE).round(3))
    print(NOTA_ROBUSTO)

### 2.3 Tabella dei risultati

`RESULTS` raccoglie una riga per modello: metriche globali sulla validation, numero di parametri
(o di valori memorizzati) e tempo di **solo training** (la valutazione è esclusa).
`registra` aggiunge un modello; se il modello è già presente, la riga viene sostituita, non duplicata.

In [5]:
RESULTS = []

def registra(modello, ris, parametri, tempo_train_s):
    """Aggiunge (o sostituisce) la riga di un modello in RESULTS usando la riga 'globale' di ris."""
    global RESULTS
    glob = ris.set_index("gruppo").loc["globale"]
    RESULTS = [r for r in RESULTS if r["modello"] != modello]
    RESULTS.append({
        "modello":        modello,
        "MAE":            glob["mae"],
        "RMSE":           glob["rmse"],
        "RMSE robusto †": glob["rmse_robusto"],
        "P90 |err|":      glob["p90_abs"],
        "P99 |err|":      glob["p99_abs"],
        "parametri":      parametri,
        "tempo_train_s":  round(tempo_train_s, 1),
    })

def tabella_risultati():
    """Mostra RESULTS come tabella, con la nota sull'RMSE robusto."""
    display(pd.DataFrame(RESULTS).round(3))
    print(NOTA_ROBUSTO)

## 3. Baseline 0 – Mediana globale

**Idea.** Prevedere per ogni flow la stessa costante, senza usare alcun input. È il *pavimento*:
qualunque modello deve fare meglio.

**Perché la mediana.** Tra tutte le costanti $c$, quella che minimizza
$\frac{1}{n}\sum_i \lvert y_i - c \rvert$ (la MAE) è la mediana; quella che minimizza l'errore
quadratico è la media. Calcoliamo entrambe per confronto.

**Come.** Mediana e media sul **train completo** (46,7 M flow, calcolate da DuckDB senza caricarle
in memoria: nessun rumore di campionamento). Previsione = mediana per ogni flow della validation.
Il tempo di training è solo quello del calcolo della mediana.

In [6]:
# --- "Training": mediana e media del train ---
t0 = time.time()
mediana, media = con.execute("""
    SELECT MEDIAN(delta_rtt), AVG(delta_rtt)
    FROM fp
    WHERE split = 'train'
""").fetchone()
t_train = time.time() - t0
print(f"train → mediana: {mediana:.3f} ms | media: {media:.3f} ms | tempo: {t_train:.1f} s")

# --- Previsione sulla validation ---
con.execute(f"""
    CREATE OR REPLACE VIEW pred_b0 AS
    SELECT delta_rtt, server_site, client_site, verso_incerto, an_durata_s,
           {mediana} AS pred
    FROM fp
    WHERE split = 'val'
""")

# --- Valutazione e registrazione ---
ris_b0 = valuta("pred_b0")
registra("B0 mediana globale", ris_b0, parametri=1, tempo_train_s=t_train)
mostra(ris_b0)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

train → mediana: 4.398 ms | media: 3.172 ms | tempo: 3.0 s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gruppo,n,MAE,RMSE,RMSE robusto †,P90 |err|,P99 |err|
0,globale,10069254,3.298,47.289,5.812,4.510,23.552
1,FNAL,215471,1.636,3.193,2.666,4.381,6.279
2,sito mancante,2122477,3.539,16.574,6.885,4.847,25.834
3,verso incerto,32662,4.570,8.801,4.790,4.423,5.541
4,flow > 1 min,6632723,3.129,58.019,6.453,4.643,27.535


† RMSE robusto: solo flow con delta_rtt in [-184.23, 58.82] ms, quantili 0.05%–99.95% del train (insieme fisso, uguale per tutti i modelli).


## 4. Baseline 1 – Mediana per tratta, con fallback gerarchico

**Idea.** Una *tratta* è la coppia di estremi di un flow (dove sta il server, dove sta il client).
Per ogni tratta calcoliamo la mediana di `delta_rtt` sul train completo; per ogni flow della validation
prevediamo la mediana della sua tratta. È una tabella di consultazione, senza parametri ottimizzati:
misura quanto spiega il solo "dove". In termini matematici stima la mediana condizionata
$\text{med}(Y \mid \text{tratta})$, cioè una funzione costante a tratti.

**Varianti (6) = 2 modi di identificare un estremo × 3 modi di ordinare la coppia.**

| | non orientata {A, B} | server → client | src → dst |
|---|---|---|---|
| siti CRIC (`MISSING` incluso) | B1 site non_orientata | B1 site server_client | B1 site src_dst |
| reti IP L2 (/24, /48) | B1 L2 non_orientata | B1 L2 server_client | B1 L2 src_dst |

- **Non orientata:** (A, B) e (B, A) sono la stessa tratta.
- **Server → client:** conta chi offre il servizio e chi lo richiede (ruolo).
- **Src → dst:** conta l'ordine con cui il monitoraggio ha registrato il flow (dipende dal software).

**Fallback (uguale per tutte le varianti).** Una tratta si usa solo se ha almeno **N = 1.000** flow
nel train, altrimenti la sua mediana è troppo rumorosa. In quel caso si usa la mediana della coppia
di operatori di rete (ASN, stesso orientamento); se anche questa ha meno di N flow, la mediana globale (B0).

**Domande.** La direzione conta? Se sì, conta il ruolo o l'ordine di registrazione?
Le reti IP (sempre presenti) battono i siti (mancanti nel ~18 % dei flow)?

In [7]:
# Espressioni SQL delle due chiavi di una coppia, per orientamento.
# {p} = prefisso di tabella ("f." oppure ""), {x} = identificatore ("site", "L2", "asn")
ORIENTAMENTI = {
    "non_orientata": ("LEAST({p}server_{x}, {p}client_{x})",
                      "GREATEST({p}server_{x}, {p}client_{x})"),
    "server_client": ("{p}server_{x}", "{p}client_{x}"),
    "src_dst":       ("{p}src_{x}",    "{p}dst_{x}"),
}

def chiavi(orient, x, p=""):
    """Le due espressioni SQL della coppia, per un orientamento e un identificatore."""
    k1, k2 = ORIENTAMENTI[orient]
    return k1.format(p=p, x=x), k2.format(p=p, x=x)

def baseline1(ident, orient, N=1000):
    """ident: 'site' o 'L2'; orient: chiave di ORIENTAMENTI; N: flow minimi di train per gruppo."""
    t0 = time.time()
    # Mediane sul train: livello 1 = coppia di siti/reti, livello 2 = coppia di ASN
    for livello, x in [(1, ident), (2, "asn")]:
        k1, k2 = chiavi(orient, x)
        con.execute(f"""
            CREATE OR REPLACE TEMP TABLE med_l{livello} AS
            SELECT {k1} AS k1, {k2} AS k2,
                   MEDIAN(delta_rtt) AS med,
                   COUNT(*)          AS n
            FROM fp
            WHERE split = 'train'
            GROUP BY ALL
            HAVING COUNT(*) >= {N}
        """)
    t_train = time.time() - t0

    # Previsione sulla validation: vince il primo livello disponibile
    a1, b1 = chiavi(orient, ident, p="f.")
    a2, b2 = chiavi(orient, "asn", p="f.")
    con.execute(f"""
        CREATE OR REPLACE VIEW pred_b1 AS
        SELECT f.delta_rtt, f.server_site, f.client_site, f.verso_incerto, f.an_durata_s,
               COALESCE(m1.med, m2.med, {mediana}) AS pred,
               CASE WHEN m1.med IS NOT NULL THEN '1 coppia {ident}'
                    WHEN m2.med IS NOT NULL THEN '2 coppia ASN'
                    ELSE '3 globale' END AS livello
        FROM fp AS f
        LEFT JOIN med_l1 AS m1 ON {a1} = m1.k1 AND {b1} = m1.k2
        LEFT JOIN med_l2 AS m2 ON {a2} = m2.k1 AND {b2} = m2.k2
        WHERE f.split = 'val'
    """)

    ris = valuta("pred_b1")
    livelli = con.execute("""
        SELECT livello,
               COUNT(*)                                  AS n,
               100.0 * COUNT(*) / SUM(COUNT(*)) OVER ()  AS pct,
               AVG(ABS(delta_rtt - pred))                AS mae
        FROM pred_b1
        GROUP BY livello
        ORDER BY livello
    """).df()
    n_param = sum(con.execute(f"SELECT COUNT(*) FROM med_l{l}").fetchone()[0] for l in (1, 2)) + 1
    return ris, livelli, n_param, t_train

### Esecuzione delle sei varianti

Per ogni combinazione di identificatore e orientamento: mediane sul train, valutazione sulla validation,
registrazione in `RESULTS`. Le tabelle di dettaglio (sottogruppi e livelli del fallback) restano in `DETTAGLI_B1`.

In [8]:
DETTAGLI_B1 = {}

for ident in ["site", "L2"]:
    for orient in ORIENTAMENTI:
        nome = f"B1 {ident} {orient}"
        ris, livelli, n_param, t_train = baseline1(ident, orient)
        DETTAGLI_B1[nome] = {"ris": ris, "livelli": livelli}
        registra(nome, ris, parametri=n_param, tempo_train_s=t_train)
        print(f"{nome:26s} MAE = {ris.loc[ris['gruppo'] == 'globale', 'mae'].iloc[0]:.3f} "
              f"| livello 1: {livelli['pct'].iloc[0]:.1f} % | {t_train:.1f} s")

tabella_risultati()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

B1 site non_orientata      MAE = 2.463 | livello 1: 99.8 % | 4.6 s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

B1 site server_client      MAE = 2.270 | livello 1: 99.7 % | 3.0 s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (6 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

B1 L2 src_dst              MAE = 2.310 | livello 1: 97.8 % | 3.6 s


,modello,MAE,RMSE,RMSE robusto †,P90 |err|,P99 |err|,parametri,tempo_train_s
0,B0 mediana globale,3.298,47.289,5.812,4.510,23.552,1,3.0
1,B1 site non_orientata,2.463,47.252,5.515,4.977,23.283,816,4.6
2,B1 site server_client,2.270,47.241,5.419,4.909,23.188,1077,3.0
3,B1 site src_dst,2.401,47.248,5.480,4.953,23.276,1029,3.0
4,B1 L2 non_orientata,2.338,47.241,5.440,4.884,23.240,1920,4.8
5,B1 L2 server_client,2.209,47.233,5.377,4.861,23.068,2166,3.3
6,B1 L2 src_dst,2.310,47.239,5.423,4.868,23.236,2140,3.6


† RMSE robusto: solo flow con delta_rtt in [-184.23, 58.82] ms, quantili 0.05%–99.95% del train (insieme fisso, uguale per tutti i modelli).


## 5. Preparazione dei dati per i modelli

I modelli (B2–B4) ricevono ogni flow come una riga di numeri: un **codice intero** per ogni input
categorico e `start_rtt`. Passi:
1. input dei modelli;
2. vocabolari (valore → codice), stimati sul train;
3. tabella `mappa` in DuckDB;
4. query di codifica;
5. insiemi di dati: `train_enc` (addestramento), `tune_enc` (early stopping e scelte), `val_enc` (risultati).

### 5.1 Input dei modelli
Orientamento server → client e reti IP L2 (le scelte migliori nella B1). Input:
- numerico: `start_rtt`;
- categorici: `ip_version`, `experiment`, `exp_act`, `application`, `hostname`,
  `server_site`, `client_site`, `server_L2`, `client_L2`, `porta_servizio`, `porta_client`,
  `verso`, `verso_incerto`.

Siti e reti L2 hanno un **vocabolario condiviso** tra server e client: lo stesso posto riceve lo stesso
codice in entrambi i ruoli. Esclusi tutti i campi noti solo alla fine del flow (durata, `end_rtt`, `an_*`).

In [ ]:
FEATURE_NUM = ["start_rtt"]
TARGET = "delta_rtt"

# vocabolario → colonne che lo usano
VOCAB_COLONNE = {
    "ip_version":     ["ip_version"],
    "experiment":     ["experiment"],
    "exp_act":        ["exp_act"],
    "application":    ["application"],
    "hostname":       ["hostname"],
    "site":           ["server_site", "client_site"],   # condiviso
    "L2":             ["server_L2", "client_L2"],       # condiviso
    "porta_servizio": ["porta_servizio"],
    "porta_client":   ["porta_client"],
    "verso":          ["verso"],
    "verso_incerto":  ["verso_incerto"],
}   

# elenco piatto di tutte le colonne categoriche
FEATURE_CAT = [c for colonne in VOCAB_COLONNE.values() for c in colonne]

print(f"{len(VOCAB_COLONNE)} vocabolari, {len(FEATURE_CAT)} colonne categoriche:")
print(FEATURE_CAT)

11 vocabolari, 13 colonne categoriche:
['ip_version', 'experiment', 'exp_act', 'application', 'hostname', 'server_site', 'client_site', 'server_L2', 'client_L2', 'porta_servizio', 'porta_client', 'verso', 'verso_incerto']


### 5.2 Vocabolari (stimati sul train)

Codici riservati: `0 = PAD` (per i modelli a token, qui inutilizzato), `1 = MISSING`,
`2 = RARE` (visto nel train ma con meno di N = 1.000 flow), `3 = UNK` (mai visto nel train).
I valori con almeno N flow ricevono i codici da 4 in su, in ordine di frequenza.
Per i vocabolari condivisi si contano insieme le occorrenze nei due ruoli.
I valori sono convertiti in testo, così numeri (`ip_version`) e vero/falso (`verso_incerto`)
si trattano come le altre categorie. I vocabolari usano solo i conteggi dei valori, non il target.

In [10]:
N_MIN = 1000
PAD, MISSING, RARE, UNK = 0, 1, 2, 3

VOCAB = {}   # nome vocabolario → {valore: codice} dei valori frequenti
VISTI = {}   # nome vocabolario → insieme di tutti i valori visti nel train

t0 = time.time()
for nome, colonne in VOCAB_COLONNE.items():
    # una SELECT per colonna, impilate una sotto l'altra con UNION ALL
    parti = [f"SELECT CAST({c} AS VARCHAR) AS valore FROM fp WHERE split = 'train'"
             for c in colonne]
    conteggi = con.execute(f"""
        SELECT valore, COUNT(*) AS n
        FROM ({" UNION ALL ".join(parti)})
        GROUP BY valore
        ORDER BY n DESC
    """).df()

    frequenti = conteggi.loc[(conteggi["n"] >= N_MIN) & (conteggi["valore"] != "MISSING"), "valore"]
    VOCAB[nome] = {v: i + 4 for i, v in enumerate(frequenti)}
    VISTI[nome] = set(conteggi["valore"])

    print(f"{nome:15s} valori nel train: {len(conteggi):6d} | con ≥ {N_MIN} flow: {len(VOCAB[nome]):5d}")

print(f"Tempo: {time.time() - t0:.1f} s")

ip_version      valori nel train:      2 | con ≥ 1000 flow:     2
experiment      valori nel train:      6 | con ≥ 1000 flow:     6
exp_act         valori nel train:     31 | con ≥ 1000 flow:    29
application     valori nel train:    168 | con ≥ 1000 flow:    56
hostname        valori nel train:    749 | con ≥ 1000 flow:   686
site            valori nel train:     91 | con ≥ 1000 flow:    83
L2              valori nel train:    720 | con ≥ 1000 flow:   340
porta_servizio  valori nel train:     47 | con ≥ 1000 flow:    46
porta_client    valori nel train:      4 | con ≥ 1000 flow:     4
verso           valori nel train:      2 | con ≥ 1000 flow:     2
verso_incerto   valori nel train:      2 | con ≥ 1000 flow:     2
Tempo: 3.9 s


### 5.3 Tabella `mappa`: valore → codice

Per ogni vocabolario e ogni valore visto nel train: il codice se frequente, `1` se `MISSING`,
`2` (RARE) se presente ma con meno di N flow. Un valore assente dalla tabella (mai visto nel train)
riceverà `3` (UNK). La tabella è salvata in DuckDB, così la codifica di milioni di righe avviene lì
e in Python arrivano solo interi.

In [11]:
righe = []
for nome in VOCAB_COLONNE:
    for valore in VISTI[nome]:
        if valore == "MISSING":
            codice = MISSING
        elif valore in VOCAB[nome]:
            codice = VOCAB[nome][valore]
        else:
            codice = RARE
        righe.append((nome, valore, codice))

mappa_df = pd.DataFrame(righe, columns=["nome", "valore", "codice"])
con.register("mappa_df", mappa_df)
con.execute("CREATE OR REPLACE TABLE mappa AS SELECT * FROM mappa_df")

print(f"Righe nella mappa: {len(mappa_df):,}")
mappa_df.groupby("nome")["codice"].agg(
    valori="count",
    rari=lambda s: (s == RARE).sum(),
)

Righe nella mappa: 1,822


,valori,rari
nome,,
L2,720,380
application,168,111
exp_act,31,2
experiment,6,0
hostname,749,63
ip_version,2,0
porta_client,4,0
porta_servizio,47,1
site,91,7


## Stato del lavoro (17 novembre 2026)

**Fatto e verificato**
- **§1 Setup.** Connessione DuckDB e vista `fp` su `flows_prepared.parquet`; conteggi per split corretti
  (train 46.730.686 · val 10.069.254 · test 10.086.352), 41 colonne.
- **§2 Valutazione.** Soglie del set tipico dal train (LO = −184,23 ms, HI = 58,82 ms; 99,90 % del train,
  99,92 % della validation); funzione `valuta` (MAE, RMSE, RMSE robusto †, P90, P99, globale e per
  sottogruppo); tabella `RESULTS` con `registra`.
- **§3 B0 – mediana globale.** Mediana del train 4,398 ms → MAE 3,298 ms (pavimento).
- **§4 B1 – mediana per tratta.** Sei varianti; la migliore è reti IP L2 + server → client:
  MAE 2,209 ms (−33 %). Conta il ruolo server/client; le reti IP battono i siti. FNAL peggiora per un
  drift tra train e validation (indagine e decisione nel notebook `03_baseline_diario`: dati invariati).
  Tutti i risultati coincidono con il notebook precedente.
- **§5 Preparazione per i modelli (in corso).**
  - 5.1 Input: 13 categorici (server/client, reti L2) + `start_rtt`.
  - 5.2 Vocabolari sul train (N = 1.000): ad es. `L2` 720 valori → 340 frequenti, `hostname` 749 → 686,
    `application` 168 → 56, `site` 91 → 83 (escluso `MISSING`).
  - 5.3 Tabella `mappa` in DuckDB: 1.822 coppie valore → codice.

**Decisioni prese**
- Loss L1 (coerente con la MAE) per LightGBM, modello lineare e rete neurale.
- Early stopping e scelta delle impostazioni sull'**ultimo 10 % del periodo di train** (`tune_enc`);
  la validation resta solo per i risultati.
- B2: modello lineare addestrato con errore assoluto (PyTorch).
- B3: embedding per campo (da confermare con il gruppo della tokenizzazione).

**Prossimi passi**
1. 5.4 Query di codifica e 5.5 insiemi `train_enc` (~3 M), `tune_enc` (~1 M), `val_enc` (10,07 M).
2. Controlli di qualità (quote RARE/UNK per campo) e salvataggio dei vocabolari in JSON per il gruppo.
3. §6 B4 LightGBM (+ diagnostica con la durata) · §7 B2 lineare L1 · §8 B3 rete neurale · §9 Conclusioni.

### 5.4 Query di codifica

`query_codificata(condizione)` costruisce la query che, per le righe di `fp` che soddisfano
`condizione`, restituisce:
- `riga`: numero progressivo, per riattaccare le previsioni ai flow;
- il target `delta_rtt` e `start_rtt`;
- un codice intero `id_<colonna>` per ogni input categorico (`LEFT JOIN` con `mappa`; valore assente → UNK);
- le colonne in chiaro che servono a `valuta` per i sottogruppi (`server_site`, `client_site`,
  `verso_incerto`, `an_durata_s`). Non sono input dei modelli.

In [12]:
def query_codificata(condizione):
    colonne_id, join = [], []
    coppie = [(n, c) for n, cols in VOCAB_COLONNE.items() for c in cols]
    for k, (nome, c) in enumerate(coppie):
        colonne_id.append(f"CAST(COALESCE(m{k}.codice, {UNK}) AS INTEGER) AS id_{c}")
        join.append(f"LEFT JOIN mappa AS m{k} "
                    f"ON m{k}.nome = '{nome}' AND m{k}.valore = CAST(f.{c} AS VARCHAR)")
    return f"""
        SELECT ROW_NUMBER() OVER () AS riga,
               f.delta_rtt, f.start_rtt,
               {", ".join(colonne_id)},
               f.server_site, f.client_site, f.verso_incerto, f.an_durata_s
        FROM fp AS f
        {" ".join(join)}
        WHERE {condizione}
    """

print(query_codificata("f.split = 'val'")[:600])   # anteprima della query generata


        SELECT ROW_NUMBER() OVER () AS riga,
               f.delta_rtt, f.start_rtt,
               CAST(COALESCE(m0.codice, 3) AS INTEGER) AS id_ip_version, CAST(COALESCE(m1.codice, 3) AS INTEGER) AS id_experiment, CAST(COALESCE(m2.codice, 3) AS INTEGER) AS id_exp_act, CAST(COALESCE(m3.codice, 3) AS INTEGER) AS id_application, CAST(COALESCE(m4.codice, 3) AS INTEGER) AS id_hostname, CAST(COALESCE(m5.codice, 3) AS INTEGER) AS id_server_site, CAST(COALESCE(m6.codice, 3) AS INTEGER) AS id_client_site, CAST(COALESCE(m7.codice, 3) AS INTEGER) AS id_server_L2, CAST(COALESCE(m8.codice, 3) AS INTEGE


### 5.5 Insiemi di dati: train, tune, validation

- **Confine del tune:** l'istante che lascia prima il 90 % dei flow del train (in ordine di tempo).
- `train_enc`: campione di ~3 M flow dal primo 90 % del train → i modelli imparano qui.
- `tune_enc`: campione di ~1 M flow dall'ultimo 10 % del train → early stopping e scelta delle impostazioni.
  Sta "nel futuro" rispetto a `train_enc`, come la validation: le scelte simulano l'uso reale.
- `val_enc`: tutta la validation → risultati, confrontabili con B0 e B1.

I campioni sono pseudo-casuali ma **riproducibili**: si tengono i flow il cui hash dell'identificativo
cade sotto una soglia (stesso campione a ogni esecuzione e su ogni computer). Il test non viene letto.

In [13]:
# --- Confine tra train vero e tune: quantile 90 % del tempo di inizio nel train ---
TUNE_START = con.execute("""
    SELECT quantile_disc(epoch(t_start), 0.9) FROM fp WHERE split = 'train'
""").fetchone()[0]
print("Inizio del tune:", pd.to_datetime(TUNE_START, unit="s", utc=True))

n_pre, n_tune = con.execute(f"""
    SELECT COUNT(*) FILTER (WHERE epoch(t_start) <  {TUNE_START}),
           COUNT(*) FILTER (WHERE epoch(t_start) >= {TUNE_START})
    FROM fp WHERE split = 'train'
""").fetchone()

# --- Soglie dei campioni (su 100.000 "caselle" dell'hash) ---
HASH = "hash(f.flow_id || CAST(f.t_start AS VARCHAR)) % 100000"
soglia_train = round(3_000_000 / n_pre  * 100_000)
soglia_tune  = round(1_000_000 / n_tune * 100_000)

cond_train = f"f.split = 'train' AND epoch(f.t_start) <  {TUNE_START} AND {HASH} < {soglia_train}"
cond_tune  = f"f.split = 'train' AND epoch(f.t_start) >= {TUNE_START} AND {HASH} < {soglia_tune}"
cond_val   = "f.split = 'val'"

t0 = time.time()
con.execute(f"CREATE OR REPLACE TABLE train_enc AS {query_codificata(cond_train)}")
con.execute(f"CREATE OR REPLACE TABLE tune_enc  AS {query_codificata(cond_tune)}")
con.execute(f"CREATE OR REPLACE TABLE val_enc   AS {query_codificata(cond_val)}")
print(f"Tempo: {time.time() - t0:.1f} s")

con.execute("""
    SELECT 'train_enc' AS tabella, COUNT(*) AS n FROM train_enc
    UNION ALL SELECT 'tune_enc', COUNT(*) FROM tune_enc
    UNION ALL SELECT 'val_enc',  COUNT(*) FROM val_enc
""").df()

Inizio del tune: 2026-10-12 05:16:06.927906990+00:00


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Tempo: 91.3 s


,tabella,n
0,train_enc,3000316
1,val_enc,10069254
2,tune_enc,1000351


# Parte 5 — Preparazione dei dati per i modelli (B2, B3, B4)

*Spiegazione di cosa fa la §5 del notebook delle baseline, perché, e cosa conviene rivedere.*
*Stato: §5.1–5.5 eseguite; i modelli (§6–8) non sono ancora stati scritti.*

---

## 0. Il quadro generale in 30 secondi

Le baseline B0 e B1 erano **tabelle di consultazione**: "per questa tratta, prevedi questa mediana".
Non serviva trasformare i dati, bastava fare `GROUP BY` in DuckDB.

I modelli veri e propri (LightGBM, modello lineare, rete neurale) invece **non sanno leggere il testo**.
Vogliono ogni flow come una riga di numeri. La §5 fa esattamente questo: prende i flow, che contengono
cose come `"CERN-PROD"`, `"xrootd"`, `"MISSING"`, e li trasforma in righe di soli numeri interi
(più `start_rtt`, che è già un numero).

```
flows_prepared.parquet  (67 M righe, colonne di testo e numeri)
        │
        │  5.2  conto quante volte compare ogni valore nel TRAIN  →  "vocabolari"
        ▼
tabella `mappa`  (1.822 righe: vocabolario, valore, codice)
        │
        │  5.4  sostituisco ogni valore di testo con il suo codice intero
        ▼
train_enc (~3 M)   tune_enc (~1 M)   val_enc (10,07 M)      ← 5.5
 "studio"        "simulazione d'esame"     "esame vero"
```

---

## 1. Perché serve tutto questo: i modelli vogliono numeri

Una variabile **categorica** è una variabile i cui valori sono etichette, non quantità:
il sito del server, l'applicazione, l'hostname. Non ha senso dire "CERN-PROD + 2" o
"la media tra xrootd e https".

Per darla a un modello si assegna a ogni etichetta un **codice intero**:

| valore        | codice |
|---------------|--------|
| `CERN-PROD`   | 4      |
| `DESY-HH`     | 5      |
| `IN2P3-CC`    | 6      |
| …             | …      |

⚠️ **Il codice è solo un nome, non una quantità.** Il codice 6 non è "più grande" del 4.
È un punto fondamentale per i passi successivi (vedi §8, punto A2): ogni modello deve sapere che quei
numeri sono **etichette**, altrimenti impara cose senza senso, del tipo "più il codice è alto, più l'RTT sale".

La tabella valore → codice si chiama **vocabolario** (stesso termine usato nella tokenizzazione:
è letteralmente la stessa idea di un vocabolario di token).

---

## 2. §5.1 — Quali input ricevono i modelli

### Cosa entra

| Tipo       | Colonne | Note |
|------------|---------|------|
| numerica   | `start_rtt` | RTT all'inizio del flow: il punto di partenza da cui si misura la variazione |
| categoriche (13) | `ip_version`, `experiment`, `exp_act`, `application`, `hostname` | *cosa* e *chi* (esperimento, attività, software, macchina) |
|            | `server_site`, `client_site` | *dove* in termini di siti CRIC (`MISSING` nel ~18 % dei flow) |
|            | `server_L2`, `client_L2` | *dove* in termini di reti IP (/24 per IPv4, /48 per IPv6): sempre presenti |
|            | `porta_servizio`, `porta_client` | porte (già raggruppate in fase di preparazione) |
|            | `verso`, `verso_incerto` | come il flow è stato registrato rispetto ai ruoli server/client |

Il **target** (ciò che si prevede) è `delta_rtt = end_rtt − start_rtt`.

### Perché proprio questi

- **Orientamento server → client e reti L2**: sono le scelte che hanno vinto nella B1
  (MAE 2,209 contro 2,270 dei siti e 2,310 di src → dst). Si usano anche i siti, perché portano
  un'informazione diversa e più "umana" (il sito raggruppa più reti).
- Le due versioni del "dove" (siti e reti) sono in parte **ridondanti**. Per LightGBM e per la rete neurale
  non è un problema; per il modello lineare vuol dire solo che i coefficienti si dividono il merito.

### Cosa è escluso, e perché è importante

Esclusi: `end_rtt`, la durata, tutte le colonne `an_*`.

Il motivo è il **leakage** (fuga di informazione): il compito è prevedere `delta_rtt` *quando il flow inizia*.
Se un modello vedesse la durata o `end_rtt`, starebbe usando il futuro per prevedere il futuro.
Sulla validation andrebbe benissimo, ma nell'uso reale sarebbe inutilizzabile, perché quelle informazioni
non esistono ancora. `end_rtt` in particolare contiene direttamente la risposta.

> Esempio intuitivo: prevedere quanto durerà una partita sapendo già a che minuto è finita.

---

## 3. §5.2 — I vocabolari

### 3.1 I quattro codici riservati

Prima dei valori "veri", i codici da 0 a 3 sono riservati a situazioni speciali:

| Codice | Nome | Quando si usa | Esempio |
|--------|------|---------------|---------|
| **0** | `PAD` | "riempitivo", per i modelli a sequenze di token. **Qui non si usa**: è riservato per compatibilità con il gruppo della tokenizzazione | — |
| **1** | `MISSING` | il dato **manca** nella sorgente | flow senza sito CRIC → `server_site = 'MISSING'` |
| **2** | `RARE` | il valore **c'è e si conosce**, ma nel train compare in **meno di 1.000 flow** | una rete /24 vista solo 40 volte |
| **3** | `UNK` | il valore **non è mai comparso nel train** (sconosciuto) | un hostname nuovo che compare solo in validation |
| 4, 5, 6, … | valori frequenti | valori con almeno 1.000 flow nel train, in ordine di frequenza (il più frequente prende 4) | `CERN-PROD` → 4 |

La distinzione **MISSING / RARE / UNK** è utile perché sono tre situazioni diverse:
"non lo so", "lo so ma l'ho visto troppo poco per fidarmi" e "non l'ho mai visto".
Il modello può imparare un comportamento diverso per ciascuna.

### 3.2 Perché una soglia di 1.000 flow (`N_MIN`)

Se un valore compare 15 volte, qualunque cosa il modello impari su di lui è quasi solo rumore:
memorizzerebbe quei 15 casi invece di imparare una regola (**overfitting**).
Raggruppare tutti i valori rari in un unico codice `RARE` dà al modello un solo "cassetto" con abbastanza
esempi per imparare qualcosa del tipo "i valori poco frequenti si comportano in media così".

La soglia **1.000** è la stessa usata nella B1 per decidere se una tratta era affidabile: c'è coerenza tra le baseline.

**Quanto pesa davvero `RARE`?** Poco. Le 380 reti L2 rare hanno ciascuna meno di 1.000 occorrenze, quindi
in tutto meno di 380 × 999 ≈ 380.000 occorrenze su ~93,5 M (46,7 M flow × 2 ruoli): **meno dello 0,4 %**.
Per `application` (111 valori rari): meno dello 0,24 % dei flow. Sono tanti *valori* diversi,
ma pochissimi *flow*.

### 3.3 I numeri del notebook, letti

| Vocabolario | Valori distinti nel train | Con ≥ 1.000 flow | Lettura |
|---|---|---|---|
| `L2` | 720 | 340 | metà delle reti sono rare, ma pesano pochissimo (vedi sopra) |
| `hostname` | 749 | 686 | quasi tutti gli host sono abbastanza frequenti |
| `application` | 168 | 56 | coda lunga di applicazioni poco usate |
| `site` | 91 | 83 | escluso `MISSING`, che ha il suo codice 1 |
| `exp_act` | 31 | 29 | |
| `porta_servizio` | 47 | 46 | |
| gli altri | 2–6 | tutti | variabili con pochi valori, nessun problema |

### 3.4 Vocabolario **condiviso** per siti e reti

`server_site` e `client_site` usano **lo stesso** vocabolario (idem `server_L2` e `client_L2`):
il CERN riceve lo stesso codice sia quando fa da server sia quando fa da client.

- **Perché:** un posto è sempre lo stesso posto. Nella rete neurale (B3) questo permette di avere una sola
  "rappresentazione" (embedding) del CERN, che impara sia dai flow in cui è server sia da quelli in cui è client:
  il doppio dei dati per la stessa cosa.
- **Come:** per decidere se un sito è frequente si sommano le sue apparizioni nei due ruoli.
- **Per LightGBM e per il modello lineare** non cambia nulla: lì le due colonne restano comunque due input distinti.

### 3.5 Cosa i vocabolari **non** usano

Solo **quante volte** compare ogni valore, mai il target. Quindi non c'è nessun rischio di "sbirciare"
la risposta costruendo i codici.

---

## 4. §5.3 — La tabella `mappa`

È semplicemente il vocabolario messo in forma di tabella dentro DuckDB, con una riga per ogni valore visto nel train:

| nome | valore | codice |
|---|---|---|
| `site` | `CERN-PROD` | 4 |
| `site` | `MISSING` | 1 |
| `site` | `SITO_PICCOLO_X` | 2 *(RARE)* |
| `application` | `xrootd` | 4 |
| … | … | … |

*(valori e codici inventati, solo per illustrare la forma)*

Righe totali: **1.822** = 720 + 749 + 168 + 91 + 47 + 31 + 6 + 4 + 2 + 2 + 2, cioè tutti i valori visti
in tutti i vocabolari. Torna.

**Perché metterla in DuckDB e non lasciarla in Python?** Perché la sostituzione testo → codice va fatta su
milioni di righe. In DuckDB è un `JOIN`, cioè un'operazione su tabelle velocissima e che non carica i dati
in memoria. In Python arrivano poi solo interi.

---

## 5. §5.4 — La query di codifica

`query_codificata(condizione)` genera una query SQL che, per ogni flow che soddisfa `condizione`,
restituisce la versione "numerica" del flow.

### Prima e dopo (esempio illustrativo)

**Prima** (una riga di `fp`):

| server_site | client_site | application | hostname | … | start_rtt | delta_rtt |
|---|---|---|---|---|---|---|
| `CERN-PROD` | `MISSING` | `xrootd` | `host-nuovo-99` | … | 12,4 | 3,1 |

**Dopo** (la stessa riga in `val_enc`):

| riga | delta_rtt | start_rtt | id_server_site | id_client_site | id_application | id_hostname | … | *server_site, client_site, verso_incerto, an_durata_s* |
|---|---|---|---|---|---|---|---|---|
| 81723 | 3,1 | 12,4 | 4 | 1 | 4 | **3** (UNK) | … | *in chiaro, solo per `valuta`* |

### Come funziona, intuitivamente

Per ognuna delle 13 colonne categoriche la query "va a cercare" il valore nella `mappa`:

- `LEFT JOIN mappa ... ON nome = 'site' AND valore = server_site`: cerca la riga della mappa giusta.
  *LEFT* vuol dire "tieni il flow anche se nella mappa non trovi niente".
- `COALESCE(codice, 3)`: se non hai trovato niente (valore mai visto nel train), metti **3 = UNK**.
- `CAST(... AS VARCHAR)`: tutto viene confrontato come testo, così `ip_version = 4` diventa `'4'`
  e `verso_incerto = true` diventa `'true'`, esattamente come quando si è costruito il vocabolario.

### Le colonne extra

- `riga`: un numero progressivo, per poter riattaccare le previsioni ai flow (attenzione: vedi §8, punto B5).
- `server_site`, `client_site`, `verso_incerto`, `an_durata_s` **in chiaro**: servono **solo** alla funzione
  `valuta` per calcolare le metriche per sottogruppo (FNAL, sito mancante, flow > 1 min…).
  **Non sono input dei modelli.** In particolare `an_durata_s` è un campo "di fine flow": se finisse per
  sbaglio tra gli input sarebbe leakage puro.

---

## 6. §5.5 — I tre insiemi di dati

### 6.1 L'idea: studio, simulazione d'esame, esame

```
 7/10 00:00                              12/10 05:16      12/10 17:12              13/10 22:18
    |────────────── TRAIN (primo 90 %) ───────|─── TUNE ────|──────── VALIDATION ────────|── test (non toccato)
    |   da qui si pesca train_enc (~3 M)       | tune_enc    |   val_enc = tutta (10,07 M)
    |                                          |  (~1 M)     |
                  "studio"                     "simulazione"           "esame"
```

| Insieme | Da dove | Quanti | A cosa serve |
|---|---|---|---|
| `train_enc` | primo 90 % (in ordine di tempo) del train | ~3 M (campione) | il modello **impara** qui |
| `tune_enc` | ultimo 10 % del train | ~1 M (campione) | **early stopping** e **scelta delle impostazioni** |
| `val_enc` | tutta la validation | 10,07 M | **risultati** da mettere in tabella accanto a B0 e B1 |

### 6.2 Perché serve un insieme di "tune" separato

Molte scelte di un modello non si imparano dai dati ma si decidono a mano: quanti alberi usare in LightGBM,
quanto regolarizzare il modello lineare, quante epoche addestrare la rete. Si chiamano **iperparametri**.

Per sceglierli bisogna provare varie opzioni e guardare quale va meglio **su dati che il modello non ha visto**.
Se lo facessimo sulla validation, la validation smetterebbe di essere un esame onesto: avremmo scelto le
impostazioni *perché* vanno bene proprio su quei dati, e il risultato finale sarebbe troppo ottimista.
È come studiare sulle domande esatte dell'esame.

**Early stopping**, in concreto: durante l'addestramento, dopo ogni giro (un albero in più, un'epoca in più)
si misura l'errore su `tune_enc`. All'inizio scende; a un certo punto il modello inizia a memorizzare il train
e l'errore sul tune risale. Ci si ferma lì.

### 6.3 Perché il tune è preso **alla fine** del train (e non a caso)

Perché nell'uso reale il modello prevede sempre il **futuro**. Il tune sta temporalmente *dopo* `train_enc`,
esattamente come la validation sta dopo il train. Se nel tempo qualcosa cambia (il drift visto su FNAL,
per esempio), il tune se ne accorge, e le scelte fatte su di lui sono più realistiche.
Un tune pescato a caso nel train sarebbe "troppo facile": conterrebbe flow quasi gemelli di quelli del train,
registrati nello stesso minuto.

Il confine (`TUNE_START` = 12/10 05:16 UTC) è il primo istante prima del quale cade il 90 % dei flow del train.

### 6.4 Perché un campione e non tutto il train

Il primo 90 % del train sono ~42 M flow. Addestrare LightGBM, e soprattutto la rete neurale, su 42 M righe
più volte (per provare impostazioni diverse) sarebbe lento. 3 M sono un compromesso tra velocità e quantità di dati.

Proporzioni effettive: `train_enc` ≈ **7,1 %** del primo 90 % del train; `tune_enc` ≈ **21 %** dell'ultimo 10 %.

### 6.5 Il campionamento con l'hash

Invece di "pesca a caso con un seme", si usa un trucco:

1. per ogni flow si calcola `hash(flow_id || t_start)`: una funzione che trasforma l'identificativo in un
   numero che *sembra* casuale ma è sempre lo stesso per lo stesso flow;
2. se ne prende il resto della divisione per 100.000: un numero tra 0 e 99.999 (le "caselle");
3. si tengono i flow con casella sotto una soglia (7.133 su 100.000 per il train, circa 21.400 per il tune).

Vantaggio: il campione **non dipende dall'ordine delle righe** né da un generatore casuale. Lo stesso flow
è dentro o fuori sempre, a ogni esecuzione. Risultato: 3.000.316 e 1.000.351 righe, praticamente i 3 M e 1 M voluti.

### 6.6 La validation è intera

`val_enc` contiene **tutti** i 10.069.254 flow della validation, esattamente le stesse righe su cui sono state
valutate B0 e B1. Quindi le MAE dei nuovi modelli saranno direttamente confrontabili con 3,298 (B0) e 2,209 (B1).
Il test non viene mai letto.

---

## 7. Riepilogo delle scelte prese

| Scelta | Perché | Alternativa possibile |
|---|---|---|
| Input: server→client, siti **e** reti L2, + `start_rtt` | migliori nella B1; siti e reti portano informazioni complementari | solo reti L2; aggiungere ASN (§8, C1) |
| Esclusi `end_rtt`, durata, `an_*` | evitare il leakage | — (non negoziabile) |
| Codici riservati PAD/MISSING/RARE/UNK | distinguere "manca", "troppo raro", "mai visto"; compatibilità con i modelli a token | un solo codice "altro" |
| Soglia `N_MIN = 1.000` sul train completo | coerente con la B1; evita di imparare rumore | soglia riferita al campione davvero usato (§8, B1) |
| Vocabolario condiviso server/client | un posto è sempre lo stesso posto; più dati per embedding | vocabolari separati per ruolo |
| Codifica in DuckDB con `mappa` + `LEFT JOIN` | veloce, senza caricare 67 M righe in memoria | codifica in pandas (molto più lenta) |
| Tune = ultimo 10 % del train | scelte e early stopping "nel futuro", come la validation | tune casuale (troppo ottimista) |
| Campioni 3 M / 1 M con hash | velocità; riproducibilità | train completo (più lento, più preciso) |
| Validation intera | confronto diretto con B0 e B1 | — |
| Loss L1 per tutti i modelli | coerente con la MAE, robusta agli outlier | L2 / Huber |

---

## 8. Cose da rivedere

Ordinate per importanza. Nessuna invalida il lavoro fatto; alcune vanno decise **prima** di scrivere §6–8.

### A. Importanti (da decidere prima dei modelli)

**A1. Dopo il tuning, riaddestrare sul train completo?**
B1 usa **tutto** il train (46,7 M flow, comprese le ultime 12 ore prima della validation). I modelli invece
imparano solo da un campione del primo 90 %. Così il confronto non è alla pari: i modelli non vedono il periodo
più recente, che è quello più simile alla validation (e con il drift su FNAL questo conta).
Pratica comune: si usa il tune per scegliere iperparametri e numero di iterazioni, poi si **riaddestra con quelle
impostazioni fisse su train + tune** e si valuta sulla validation. Da decidere e scrivere nel notebook.

**A2. Come ogni modello deve leggere i codici interi**
I codici sono etichette (§1). Concretamente:
- **B4 LightGBM**: dichiarare le 13 colonne come `categorical_feature`. Altrimenti le tratta come numeri
  ordinati, e "hostname 700 > hostname 5" non vuol dire nulla.
  Nota: LightGBM ha per default `min_data_per_group = 100`, quindi ignora di fatto le categorie con pochissime
  righe nel campione (si collega a B1 qui sotto).
- **B2 modello lineare**: **non** si possono passare i codici come numeri. Serve un *one-hot*
  (una colonna 0/1 per ogni valore) o, in PyTorch, l'equivalente: un `nn.Embedding(n_valori, 1)` per campo, i cui
  valori si sommano. Ogni valore ottiene così un suo coefficiente.
- **B3 rete neurale**: `nn.Embedding` per campo. Per i vocabolari condivisi, la **stessa** tabella per server e client.
  Dimensione di ogni tabella = 4 + numero di valori frequenti (es. `L2`: 4 + 340 = 344).

**A3. Verificare i valori NULL**
Il codice gestisce il caso in cui il valore mancante sia la **stringa** `'MISSING'`. Ma se in qualche colonna
il mancante fosse un vero `NULL`, il `JOIN` con la mappa non lo troverebbe mai (in SQL `NULL = NULL` non è vero),
e quel flow finirebbe in **UNK (3)** invece che in **MISSING (1)**: due situazioni diverse confuse tra loro.
Controllo da fare, una query:
```sql
SELECT COUNT(*) FILTER (WHERE application IS NULL)   AS null_application,
       COUNT(*) FILTER (WHERE hostname IS NULL)      AS null_hostname,
       COUNT(*) FILTER (WHERE server_L2 IS NULL)     AS null_server_L2
       -- ... una riga per ciascuna delle 13 colonne
FROM fp;
```
Se tutto è 0, nessun problema. Altrimenti basta sostituire i `NULL` con `'MISSING'` prima della codifica.

### B. Da tenere presenti (impatto moderato)

**B1. La soglia 1.000 è contata sul train completo, ma i modelli ne vedono il 7 %**
Un valore con 1.000 flow nel train ne avrà in media solo ~70 in `train_enc`. Peggio: un valore che compare solo
nelle ultime 12 ore ha un codice "frequente" ma **zero** esempi in `train_enc`, quindi il suo embedding resta
casuale e LightGBM non ha niente da cui imparare. Opzioni: accettarlo consapevolmente (sono pochi flow), oppure
contare la soglia sulla parte di train effettivamente usata per addestrare. Il controllo delle quote RARE/UNK
già previsto nei prossimi passi è il posto giusto per misurarlo.

**B2. Il vocabolario include il periodo di tune, quindi nel tune non esiste UNK**
Visto che i vocabolari sono costruiti su tutto il train, ogni valore del tune è "conosciuto" per costruzione.
Il tune quindi **non simula** la comparsa di valori nuovi, che invece in validation c'è. Effetto probabilmente
piccolo, ma rende il tune un po' più facile della validation. Si risolve automaticamente se si costruiscono i
vocabolari solo sul primo 90 % (e poi, con il riaddestramento di A1, sul train completo).

**B3. Il tune copre solo ~12 ore, tutte di giorno**
Dal 12/10 05:16 al 12/10 17:12 UTC: mattina e pomeriggio europei, nessuna notte. Se `delta_rtt` ha un ciclo
giornaliero (traffico diurno vs notturno), le scelte fatte sul tune potrebbero non valere per la notte, che invece
è nella validation (~29 ore). Non è grave, ma conviene saperlo, ed eventualmente controllare la MAE per ora del giorno.

**B4. `hash()` di DuckDB potrebbe cambiare tra versioni**
Il campione è identico a ogni esecuzione **con la stessa versione di DuckDB** (qui la 2.0.0). La documentazione
di DuckDB non garantisce che `hash()` resti uguale tra versioni, quindi un compagno con un'altra versione potrebbe
ottenere un campione diverso. Alternativa stabile ovunque: usare `md5`, per esempio
```sql
md5_number(f.flow_id || CAST(f.t_start AS VARCHAR)) % 100000
```
(da verificare che la funzione esista nella vostra versione; in alternativa si possono prendere le prime cifre
esadecimali di `md5(...)`).

**B5. `riga` non è riproducibile tra esecuzioni**
`ROW_NUMBER() OVER ()` senza `ORDER BY` numera le righe nell'ordine in cui DuckDB le produce, che con 8 thread e
`preserve_insertion_order = false` cambia da un'esecuzione all'altra. Dentro una stessa tabella già creata va bene
(le previsioni si riattaccano alla stessa `val_enc`). Ma se salvate previsioni su file e le ricaricate in un'altra
sessione, `riga` non corrisponde più agli stessi flow. Soluzione semplice: `ROW_NUMBER() OVER (ORDER BY f.t_start, f.flow_id)`,
oppure tenere `flow_id` e `t_start` come chiave.

**B6. Codici a pari frequenza: ordine non garantito**
I codici sono assegnati per frequenza decrescente (`ORDER BY n DESC`). Se due valori hanno esattamente lo stesso
conteggio, quale prende il codice più basso può cambiare tra esecuzioni. Visto che i vocabolari verranno salvati in
JSON e condivisi con il gruppo, conviene renderli deterministici: `ORDER BY n DESC, valore`.

### C. Domande aperte (scelte di modellazione, non errori)

**C1. Manca l'ASN tra gli input.** Nella B1 la coppia di ASN era il livello di riserva quando la tratta era troppo
rara. Nei modelli, per un flow con rete L2 rara e sito `MISSING`, non c'è nessuna informazione di "dove" a grana più
grossa. Aggiungere `server_asn` / `client_asn` (con vocabolario condiviso) replicherebbe la gerarchia della B1.
Riguarda pochi flow, ma è coerente con quanto fatto finora.

**C2. Scala di `start_rtt` per B2 e B3.** Per LightGBM non importa. Per il modello lineare e la rete neurale un input
con coda lunga (RTT da pochi ms a centinaia) va trasformato: per esempio `log(start_rtt)` e poi standardizzazione,
con media e deviazione calcolate **solo su `train_enc`**.

**C3. Costruire `X` con l'elenco esplicito delle colonne.** Le tabelle `*_enc` contengono anche colonne che **non**
sono input (`riga`, `delta_rtt`, le quattro colonne in chiaro, tra cui `an_durata_s`). Usare sempre
`["start_rtt"] + ["id_" + c for c in FEATURE_CAT]`, mai "tutte tranne il target".

**C4. `verso` e `verso_incerto` sono noti all'inizio del flow?** Dipende da come sono stati costruiti nella
preparazione. Se l'assegnazione dei ruoli usa solo porte e indirizzi, va bene; se usasse informazioni di fine flow
(byte trasferiti, durata…) sarebbe leakage. Basta una verifica nel notebook di preparazione.

**C5. Ora del giorno come input?** `t_start` è noto all'inizio del flow, quindi l'ora (o il giorno della settimana)
sarebbe un input legittimo. Da decidere col gruppo, anche in base a cosa farà la tokenizzazione.

---

## 9. Mini-glossario

| Termine | Significato |
|---|---|
| **Variabile categorica** | variabile i cui valori sono etichette (sito, applicazione), non quantità |
| **Vocabolario** | tabella valore → codice intero |
| **Embedding** | vettore di numeri imparato dalla rete neurale per ogni codice: la "rappresentazione" di quel valore |
| **One-hot** | trasformare una categorica in tante colonne 0/1, una per valore |
| **Leakage** | usare come input un'informazione che nell'uso reale non sarebbe disponibile |
| **Overfitting** | memorizzare i casi del train invece di imparare regole generali |
| **Iperparametro** | impostazione del modello scelta a mano (n. di alberi, regolarizzazione, epoche) |
| **Early stopping** | fermare l'addestramento quando l'errore su dati non visti smette di scendere |
| **Drift** | cambiamento nel tempo del comportamento dei dati |
| **Hash** | funzione che trasforma un valore in un numero dall'aspetto casuale ma sempre uguale per lo stesso input |
| **LEFT JOIN** | unione di tabelle che tiene tutte le righe di sinistra anche se a destra non c'è corrispondenza |
| **COALESCE(a, b)** | "a se esiste, altrimenti b" |
| **MAE** | errore assoluto medio: di quanti ms si sbaglia in media |